# 03 — Betting-Edge Analysis (historical / illustrative only)

> **Caveats — please read these first.**
> - **Historical only.** This notebook backtests on past matches against past (closing) odds. It is **not** a real-money strategy and there is **no live-odds ingestion anywhere in this repo**.
> - **Closing odds, not in-play.** Lines move; closing prices are the post-information equilibrium and the strongest available benchmark. Pre-game and in-play odds would tell a different story.
> - **Bookmaker hold is removed but not modelled.** We strip the overround to read implied probabilities off the price; we do *not* model risk limits, account restrictions, or the price you'd actually receive after stake size.
> - **Transaction costs ignored.** Commissions, deposit / withdrawal friction, time-value of money — none accounted for.
> - **Small-sample variance.** Even a positive-edge strategy can lose money over a few hundred matches; confidence intervals are wide; do not over-interpret.
> - **No real-money implication.** This is portfolio work showing model-vs-market evaluation under proper scoring rules; the analysis is **illustrative**, not advice.

A thin narrative over `matchodds.modeling.betting`: turn closing odds into overround-removed implied probabilities, measure per-match edge, and backtest a flat-stake positive-EV strategy on **out-of-sample** matches. All logic lives in the `matchodds` package; the cells only import and call it.

> Run `make data && make features` first, or point `MATCHODDS_DATA_DIR` at the committed sample.

## Setup

Build the feature table in-memory and keep rows that carry usable closing odds (all three odds finite and strictly > 1.0 — the rejection rule `betting.implied_probabilities` enforces).

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matchodds.config import settings
from matchodds.features import pipeline
from matchodds.modeling import betting, calibration, metrics
from matchodds.modeling.cv import TimeOrderedSplit
from matchodds.modeling.logistic import LogisticModel

ODDS = ["odds_home", "odds_draw", "odds_away"]

table = pipeline.build(write=False)
valid = table[ODDS].notna().all(axis=1) & (table[ODDS] > 1.0).all(axis=1)
table = table[valid].reset_index(drop=True)
metadata = json.loads((settings.models_dir / "v1.metadata.json").read_text())
print(f"{len(table):,} matches with usable closing odds | {table['date'].min()} -> {table['date'].max()}")
print(f"selected model (per metadata): {metadata['selected_model']} | calibration: {metadata['calibration_method']}")
table[["date", "league", "home", "away", "result", *ODDS]].head()

## Overround removal

A bookmaker's decimal odds carry a margin — the **overround** or **hold** — so the raw inverse probabilities `1 / odds_i` sum to **more than** 1. `betting.implied_probabilities` divides each row by its sum, producing normalised implied probabilities that sum to exactly 1 (the standard "remove the bookmaker hold" transform).

In [ ]:
sample = table.head(5)
sample_odds = sample[ODDS].to_numpy(dtype=float)
implied = betting.implied_probabilities(sample_odds)
raw_sum = (1.0 / sample_odds).sum(axis=1)

pd.DataFrame(
    {
        "H_odds": sample_odds[:, 0],
        "D_odds": sample_odds[:, 1],
        "A_odds": sample_odds[:, 2],
        "overround": raw_sum - 1.0,
        "P(H)": implied[:, 0],
        "P(D)": implied[:, 1],
        "P(A)": implied[:, 2],
    }
).round(3)

## Model vs book — out-of-sample edge

For each fold of `TimeOrderedSplit`, refit the **selected** model (logistic, per `models/v1.metadata.json`) on the fold's training slice and predict on the strictly-later test slice. Calibration uses the same `calibration.calibrate(method="sigmoid", cv=safe_calibration_cv(train))` path the shipped artifact does (the bake-off method from `train.py`). This produces **out-of-sample** predictions for every match in any test fold — never in-sample, never leaking the future. The committed `models/v1.joblib` is deliberately **not** used here: it was fit on all data, so scoring it against any of its own training rows would be in-sample.

Per-match edge is then `betting.edge(predictions, market_odds)`. Positive values mean the model gives the outcome a higher probability than the (de-overrounded) market does.

In [ ]:
splitter = TimeOrderedSplit(table["date"])
predictions = np.zeros((len(table), 3))
in_holdout = np.zeros(len(table), dtype=bool)
for train_idx, test_idx in splitter.split():
    cv = calibration.safe_calibration_cv(table.iloc[train_idx])
    if cv is None:
        continue
    model = calibration.calibrate(LogisticModel(), table.iloc[train_idx], method="sigmoid", cv=cv)
    predictions[test_idx] = model.predict_proba(table.iloc[test_idx])
    in_holdout[test_idx] = True

market_odds = table[ODDS].to_numpy(dtype=float)
edges = betting.edge(predictions[in_holdout], market_odds[in_holdout])
print(f"out-of-sample predictions: {int(in_holdout.sum()):,} matches across {splitter.get_n_splits()} folds")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(edges.ravel(), bins=60, color="steelblue", alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", label="model = market")
ax.set_xlabel("Edge (model probability − implied market probability)")
ax.set_ylabel("Count")
ax.set_title("Per-outcome edge distribution (out-of-sample)")
ax.legend()
fig.tight_layout()

## Flat-stake positive-EV backtest

For each match, pick the outcome with the largest expected value and stake one unit on it iff that EV exceeds the threshold (the rule baked into `betting.backtest`). Three thresholds (0% / 2% / 5%) trade off bet count vs. per-bet variance:

In [ ]:
holdout_dates = table.loc[in_holdout, "date"].to_numpy()
oop_preds = predictions[in_holdout]
oop_odds = market_odds[in_holdout]
oop_results = metrics.encode_labels(table.loc[in_holdout, "result"])

summary_rows = []
for threshold in (0.00, 0.02, 0.05):
    summary = betting.backtest(oop_preds, oop_odds, oop_results, edge_threshold=threshold)
    summary_rows.append({"edge_threshold": f"{threshold:.0%}", **summary})
pd.DataFrame(summary_rows).set_index("edge_threshold")[
    ["n_matches", "n_bets", "n_wins", "win_rate", "pnl", "roi"]
].round(4)

In [ ]:
# Per-match P&L at the default 0% threshold, for the cumulative path plot.
ev = betting.expected_value(oop_preds, oop_odds)
picks = ev.argmax(axis=1)
rows = np.arange(len(ev))
bet_mask = ev[rows, picks] > 0.0
win_mask = bet_mask & (picks == oop_results)
profit = np.where(bet_mask, np.where(win_mask, oop_odds[rows, picks] - 1.0, -1.0), 0.0)
cumulative = profit.cumsum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(holdout_dates, cumulative, color="steelblue")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Match date")
ax.set_ylabel("Cumulative PnL (units of stake)")
ax.set_title(f"Flat-stake EV>0 backtest — {int(bet_mask.sum()):,} bets, time-ordered")
fig.tight_layout()

## Takeaways

- **Closing odds are a very hard benchmark.** They are the post-information equilibrium price — already reflecting public knowledge and market consensus — so a simple feature-engineered logistic, even calibrated, is unlikely to beat them by much (or at all) on average. Expect ROI to sit near or below zero across reasonable thresholds; treat any large positive sample-period ROI with suspicion (it reflects the data window as much as model edge).
- **The bookmaker's overround is small in absolute terms** (a few percent per match) but baked into the price you'd pay; `betting.implied_probabilities` is the right denominator for "is the model better than the market?".
- **Raising the edge threshold filters out marginal bets** but doesn't transform a negative-edge strategy into a positive one — it just reduces variance and bet count.
- **What this exercise actually demonstrates** is that the shipped model is *honest* (calibrated probabilities + proper-scoring evaluation, per the model card), not that it's a profit engine. Beating the closing market is a much harder problem than producing well-calibrated probabilities, and explicitly out of scope for this repo.

**Repeat caveats.** Historical / illustrative only. No live odds anywhere in this repo. No real-money implication. The shipped model (Epic 04 → Epic 06.5) targets honest probabilities, not betting profit; this notebook just measures the model against the market in a way that maps to one common framing.